# Machine Learning Project: Country-Year Panel Analysis with PCA, K-Means, and t-SNE

This notebook demonstrates how to preprocess a World Bank-style dataset (countries, indicators, yearly values), handle missing data, and apply machine learning techniques (PCA, K-Means clustering, and t-SNE visualization).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

pd.set_option('display.max_columns', 120)
RANDOM_STATE = 42

## 1. Load and reshape dataset

In [ ]:
path = 'WDICSV.csv'
raw = pd.read_csv(path)

year_cols = [c for c in raw.columns if c.isdigit()]
meta_cols = ['Country Name','Country Code','Indicator Name','Indicator Code']

long = raw.melt(id_vars=meta_cols, value_vars=year_cols, var_name='Year', value_name='Value')
long['Year'] = pd.to_numeric(long['Year'], errors='coerce').astype('Int64')
long['Value'] = pd.to_numeric(long['Value'], errors='coerce')

panel = long.pivot_table(index=['Country Name','Country Code','Year'],
                       columns='Indicator Code',
                       values='Value',
                       aggfunc='mean').sort_index()
panel.head()

## 2. Coverage-based filtering

In [ ]:
def describe_missing(df):
    total = df.size
    missing = df.isna().sum().sum()
    return {
        'rows': df.shape[0],
        'features': df.shape[1],
        'missing_pct': float(missing/total) if total>0 else np.nan
    }

ROW_THRESH, FEAT_THRESH = 0.7, 0.7  # adjust thresholds as needed

filtered = panel.copy()
feat_cov = filtered.notna().mean(axis=0)
filtered = filtered.loc[:, feat_cov >= FEAT_THRESH]
row_cov = filtered.notna().mean(axis=1)
filtered = filtered.loc[row_cov >= ROW_THRESH]
describe_missing(filtered)

## 3. Imputation strategy

In [ ]:
def interpolate_country(df_country):
    years = df_country.index.get_level_values('Year')
    df_country = df_country.copy()
    df_country['__Year__'] = years
    df_country = df_country.sort_values('__Year__').drop(columns='__Year__')
    return df_country.interpolate(method='linear', limit_direction='both', axis=0)

imputed = (filtered.groupby(level='Country Code', group_keys=False)
                    .apply(interpolate_country))
country_medians = imputed.groupby(level='Country Code').transform('median')
imputed = imputed.fillna(country_medians)
imputed = imputed.fillna(imputed.median(axis=0))
describe_missing(imputed)

## 4. Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(imputed.values)

## 5. PCA for dimensionality reduction

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
print('PCA explained variance ratio:', pca.explained_variance_ratio_)

## 6. K-Means clustering

In [ ]:
K = 4  # set after inspecting elbow method
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
clusters = kmeans.fit_predict(X_pca)

plt.figure(figsize=(6,4))
plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap='tab10')
plt.title('K-Means Clusters (PCA space)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()

## 7. t-SNE visualization

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', n_iter=1500, random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(7,5))
plt.scatter(X_tsne[:,0], X_tsne[:,1], c=clusters, cmap='tab10', s=15)
plt.title('t-SNE visualization of country-year data')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.show()